# Workflow Automation and Web Data Ingestion with Python

Lab notebook:
- `LabSession/Lab_Web_Automation_Data_Ingestion_STUDENT.ipynb`

Install dependencies if needed:

```bash
python -m pip install -r IntermediatePython/requirements_web_automation.txt
```


# Reference Documentation

- [Requests documentation](https://docs.python-requests.org/en/stable/): HTTP requests, responses, headers, timeouts, sessions, and errors.
- [Beautiful Soup documentation](https://beautiful-soup.readthedocs.io/en/latest/): parsing HTML/XML, search methods, CSS selectors, text extraction, and attributes.
- [Selenium Python API documentation](https://www.selenium.dev/selenium/docs/api/py/): WebDriver, WebElement, locators, waits, and browser automation APIs.
- [Selenium documentation](https://www.selenium.dev/documentation/): broader Selenium concepts, WebDriver setup, and cross-language guidance.

## 0) Notebook Setup

Before running the examples, the notebook needs a few common imports, a place to write output files, and access to the local demo pages. The examples use local HTML files so the lesson is repeatable and does not depend on a public website changing during class.

Install dependencies if needed:

```bash
python -m pip install -r requirements_web_automation.txt
```

Selenium also needs a browser. Recent Selenium versions can manage ChromeDriver automatically, but the machine still needs a compatible browser installed.


In [ ]:

from pathlib import Path
import csv
import json
import logging
import socket
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urljoin

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("web-ingestion-lesson")

BASE_DIR = Path.cwd()
DEMO_CANDIDATES = [BASE_DIR / "web_automation_demo", BASE_DIR / "IntermediatePython" / "web_automation_demo"]
DEMO_DIR = next((path.resolve() for path in DEMO_CANDIDATES if path.exists()), None)

if DEMO_DIR is None:
    raise FileNotFoundError(
        "Could not find web_automation_demo. Expected it under IntermediatePython/web_automation_demo."
    )

DATA_DIR = BASE_DIR / "data" / "web_ingestion"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
LOG_DIR = BASE_DIR / "logs"

for directory in [RAW_DIR, PROCESSED_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Demo directory:", DEMO_DIR)
print("Processed output directory:", PROCESSED_DIR)


## 1) Why Automation Matters

Automation is useful when a task has to be done repeatedly and the steps are predictable. A human can still decide what should happen, but the computer can handle the routine parts: downloading, checking, transforming, saving, and reporting.

Manual repetitive work is expensive because it consumes attention, is easy to forget, and often fails silently.

Common examples:
- Downloading reports every morning.
- Extracting website table data.
- Validating files before loading them.
- Sending summaries to stakeholders.
- Loading cleaned data into CSV files, databases, or data lakes.

**Automation goal:** make repeated work consistent, observable, and easier to rerun.


## 2) What Is Workflow Automation?

A workflow is a sequence of tasks that runs for a reason and produces a result. In practice, a useful automation has a trigger, ordered tasks, dependencies between those tasks, outputs, and some way to know whether the work succeeded.

Workflow automation is the design and execution of repeatable tasks with clear triggers, dependencies, outputs, and monitoring.

```text
Trigger -> Extract -> Validate -> Transform -> Load -> Notify
```

Core pieces:
- **Trigger:** what starts the workflow.
- **Task:** one unit of work.
- **Dependency:** what must finish before another task starts.
- **Output:** file, database row, report, message, model input, or API response.
- **Monitoring:** how we know if it worked.


## 3) Scripts vs Workflows vs Pipelines

The words script, workflow, and pipeline are related, but they are not exactly the same. A script is usually one piece of code. A workflow describes how multiple tasks run together. A pipeline is a workflow focused on moving or transforming data.

| Concept | Meaning | Example |
| --- | --- | --- |
| Script | One piece of code that performs a task | `download_report.py` |
| Scheduled job | A script or command run at a specific time | Run every weekday at 8 AM |
| Workflow | Ordered set of tasks with dependencies | Download -> validate -> save -> notify |
| Pipeline | Structured movement and transformation of data | Raw HTML -> parsed rows -> cleaned CSV |
| Orchestrator | System that schedules, runs, monitors, and retries workflows | Airflow, Prefect, Dagster |


## 4) Common Automation Scenarios

Most automation work starts with an ordinary repeated task. The examples below are common in data, analytics, and operations teams, and each one can begin as a small Python script before becoming a larger workflow.

| Scenario | Practical task |
| --- | --- |
| File ingestion | Pick up CSV files from a folder and validate them |
| Email attachment processing | Download invoices and store metadata |
| Web scraping | Extract public table or product data |
| API ingestion | Pull JSON data from a service |
| Database updates | Upsert records after validation |
| Report generation | Create weekly summaries |
| Data validation | Check schema, nulls, duplicates, and ranges |
| Notifications | Send a message when a workflow fails |


## 5) Core Workflow Concepts

Reliable automation depends on operational habits, not only Python syntax. The concepts in this table are the vocabulary used to discuss whether a workflow is safe to rerun, easy to debug, and clear enough for someone else to maintain.

| Concept | Meaning |
| --- | --- |
| Trigger | Event or schedule that starts work |
| Task | One unit of work |
| Dependency | Required order between tasks |
| Input/output | What each task consumes and produces |
| Retry | Try again after temporary failure |
| Logging | Record what happened |
| Error handling | Decide what to do when something fails |
| Scheduling | Run at a specific time or interval |
| Idempotency | Safe to rerun without corrupting results |
| Monitoring | Detect success, failure, and unusual behavior |


## 6) Automation Framework Categories

Different automation tools solve different problems. Instead of choosing a tool because it is popular, first identify whether the task needs local execution, scheduling, orchestration, cloud infrastructure, browser interaction, or user-interface automation.

| Category | Tools | Best for |
| --- | --- | --- |
| Local scripts | Python, Bash | Small automations and prototypes |
| Scheduling | cron, Windows Task Scheduler | Time-based execution |
| Workflow orchestrators | Airflow, Prefect, Dagster | Multi-step workflows with monitoring |
| Cloud automation | Azure Data Factory, AWS Glue, Google Cloud Workflows | Managed cloud pipelines |
| Browser automation | Selenium, Playwright | Browser interaction and dynamic pages |
| RPA tools | UiPath, Power Automate | GUI-heavy business processes |


## 7) Where Web Automation Fits

Web automation is one way to collect data when the source system is a website. It usually covers only the extraction part of the work; after extraction, the data still needs validation, storage, logging, and sometimes notification.

Web automation is one kind of workflow automation. It is useful when the source system is a website rather than a clean API or direct database.

Use cases:
- Extracting product prices.
- Downloading public reports.
- Collecting table data.
- Automating repetitive browser interactions.
- Monitoring website changes.

```text
Website -> Extractor -> Parser -> Validator -> Storage -> Report or notification
```


## 8) Web Data Ingestion Options

When data appears on a website, the first question is not “Which scraper should I use?” The first question is “Where does the data actually live?” It may come from an official API, raw HTML, a background JSON request, or a browser-rendered DOM.

| Option | Description | Typical tool |
| --- | --- | --- |
| Official API | Structured endpoint designed for programs | `requests` |
| HTTP requests | Download raw HTML, CSV, JSON, or files | `requests` |
| HTML parsing | Extract data from HTML text | Beautiful Soup |
| Browser automation | Interact with rendered pages | Selenium |
| Manual export automation | Automate download buttons or forms | Selenium or RPA |

**Default rule:** prefer official APIs when available and allowed. They are usually more stable, documented, and respectful of the source system.


## 9) How Web Pages Are Structured

A web page is not a single object. It is built from several layers that the browser combines into what we see on screen. For scraping, the goal is to find the layer where the target data lives before choosing a tool.

Most pages are built from four practical layers:
- **HTML:** structure and content.
- **CSS:** styling and layout.
- **JavaScript:** behavior, interactivity, and dynamic content.
- **Network requests:** background calls that may fetch data from APIs.

A useful mental shortcut is: the browser requests a URL, receives HTML/CSS/JavaScript, JavaScript may fetch more data, and the browser renders the final DOM. The scraper’s job is to find where the data lives in that chain.

A scraper must understand where the data lives:
- In the initial HTML?
- Rendered later by JavaScript?
- Loaded from a background API?
- Hidden behind authentication?
- Generated after clicking or scrolling?

```text
Browser
  -> requests page
  -> receives HTML, CSS, JavaScript
  -> JavaScript may request more data
  -> browser renders final DOM
```


## 10) HTML Elements

HTML elements are the building blocks that scraping tools search through. A page may look visual in the browser, but underneath it is made of tags, text, and attributes that can be selected and extracted.

HTML is made of elements.

```html
<h1>Product Catalog</h1>

<div class="product-card" id="product-001">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <button>Add to cart</button>
</div>
```

- `<h1>` is a heading element.
- `<div>` is a generic container.
- `<h2>` is a product title.
- `<p>` is a paragraph.
- `<button>` is an interactive element.
- `class` groups similar elements, which is why repeated products often share the same class.
- `id` identifies a unique element, so it should normally appear once on a page.
- Text content is the visible text inside tags.
- Attributes provide extra information.


## 11) Tags, Attributes, and Text

Data can appear either as visible text or as an attribute value. For example, the label of a link is text, while the destination URL is usually stored in the `href` attribute.

Breakdown:

```html
<a href="https://example.com/product/1" class="product-link">
  View product
</a>
```

| Part | Value |
| --- | --- |
| Tag | `a` |
| Attribute | `href` |
| Attribute value | `https://example.com/product/1` |
| Class | `product-link` |
| Text | `View product` |

Python extraction:

```python
link_text = element.get_text(strip=True)
link_url = element.get("href")
```

Data may be in visible text or inside attributes. Product names are often text, URLs are usually `href`, image URLs are usually `src`, and metadata may live in `data-*` attributes.


## 12) Common HTML Elements Used in Scraping

Some HTML tags appear again and again in scraping work. Headings often hold titles, links hold URLs, tables hold rows and cells, and generic containers such as `div` and `span` often hold modern website layouts.

| Element | Meaning | Common scraping use |
| --- | --- | --- |
| `h1`, `h2`, `h3` | Headings | Titles, section names |
| `p` | Paragraph | Descriptions |
| `a` | Link | URLs, navigation |
| `img` | Image | Image sources |
| `table` | Table | Structured tabular data |
| `tr` | Table row | Row extraction |
| `td` | Table cell | Cell extraction |
| `ul`, `ol` | Lists | Menus, item lists |
| `li` | List item | Individual records |
| `div` | Generic container | Cards, sections, layout blocks |
| `span` | Inline container | Prices, labels, small fields |
| `form` | Form | Search and login forms |
| `input` | Input field | Form automation |
| `button` | Button | Click actions |


## 13) HTML Hierarchy and Nesting

The DOM is a tree. Elements can be parents, children, or siblings, and a reliable scraper uses that structure instead of treating the page as a flat list of text. The safest pattern is to select each repeated parent container first, then search inside it.

HTML elements have parent-child relationships.

```html
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a href="/products/mouse">Details</a>
</div>
```

The `div` is the parent container. The `h2`, `p`, and `a` are child elements. If there were several product cards, those cards would be siblings in the DOM tree.

The scraper’s logic should usually be: find all cards first, then extract the fields from each card. This keeps each product record together even if another product is missing a field.

```python
card = soup.select_one(".product-card")

name = card.select_one("h2").get_text(strip=True)
price = card.select_one(".price").get_text(strip=True)
url = card.select_one("a").get("href")
```


## 14) DOM: The Browser's Version of the Page

The DOM is the browser’s live version of the page. Requests and Beautiful Soup see the initial HTML text. Selenium sees the page after the browser has loaded it and JavaScript has had a chance to change it.

DOM means Document Object Model.

- The DOM is the browser's live representation of the page.
- JavaScript can modify the DOM after the first HTML is loaded.
- Selenium interacts with the rendered browser DOM.
- Beautiful Soup parses HTML text, not a live browser DOM.

| Concept | Requests + Beautiful Soup | Selenium |
| --- | --- | --- |
| Sees initial HTML | Yes | Yes |
| Sees JavaScript-rendered content | No, unless already in HTML | Yes |
| Can click buttons | No | Yes |
| Can scroll | No | Yes |
| Can fill forms | Not directly | Yes |
| Faster | Usually yes | Usually no |
| Realistic browser behavior | No | Yes |


## 15) Inspecting a Web Page

Inspection is the bridge between looking at a page and writing automation code. Before writing selectors, inspect the element, move outward to the repeated parent container, and check whether the data is in text, attributes, HTML, or a network response.

A practical inspection workflow:

1. Open the page.
2. Right-click the target data.
3. Click Inspect.
4. Find the surrounding HTML.
5. Identify stable selectors.
6. Test selectors in DevTools search or console.

Look for:
- Unique `id`.
- Meaningful `class`.
- Parent container.
- Repeated card structure.
- `href`, `src`, or `data-*` attributes.
- Table structure.
- Pagination buttons.
- Search forms.


## 16) Selectors: How We Locate Elements

Selectors are the patterns we use to locate elements. CSS selectors are a good default because they are readable and transfer across browser DevTools, Beautiful Soup, and Selenium.

Selectors are patterns used to find elements.

```css
h1
.product-card
#main-content
a[href]
img[src]
.product-card .price
div.product-card h2
```

- `h1` selects all `<h1>` elements.
- `.product-card` selects elements with class `product-card`.
- `#main-content` selects the element with id `main-content`.
- `a[href]` selects links that have an `href` attribute.
- `.product-card .price` selects elements with class `price` inside `.product-card`.

CSS selectors work in Beautiful Soup, Selenium, browser DevTools, and many other tools.


## 17) CSS Selectors vs XPath

CSS selectors and XPath are both ways to locate elements. For most beginner scraping tasks, CSS selectors are easier to read and maintain. XPath becomes useful when you need complex hierarchy logic or text-based matching.

Selenium supports multiple locator strategies:
- ID
- Name
- Class name
- Tag name
- CSS selector
- XPath
- Link text
- Partial link text

| Selector type | Example | When useful |
| --- | --- | --- |
| CSS selector | `.product-card .price` | Clean, readable, common scraping |
| XPath | `//div[@class='product-card']//p` | Complex hierarchy or text matching |
| ID | `#search-box` | Stable unique elements |
| Class | `.price` | Repeated components |
| Link text | `View details` | Simple link automation |

```python
driver.find_element(By.ID, "search-box")
driver.find_elements(By.CLASS_NAME, "product-card")
driver.find_elements(By.CSS_SELECTOR, ".product-card .price")
driver.find_element(By.XPATH, "//button[contains(text(), 'Search')]")
```


## 18) Good vs Bad Selectors

A selector should describe the data, not the page layout by accident. Stable selectors use meaningful classes, IDs, attributes, and nearby parent containers. Fragile selectors often come from copying a full path from DevTools without understanding it.

Good selectors are stable, meaningful, and close to the data.

```css
.product-card
.product-card .price
table.sales-report tbody tr
a.product-link
button[type='submit']
```

Fragile selectors often come from generated classes or full paths copied from the browser. They break when layout changes:

```css
div > div > div > div:nth-child(3)
.css-1829abc
body > main > section > div:nth-child(2)
```

Why fragile selectors break:
- Layout changes.
- Auto-generated class names.
- New elements inserted.
- Ads or banners changing the page structure.
- A/B testing.


## 19) Requests: Getting the Raw HTML

Requests works at the HTTP response layer. It asks a server for a resource and gives your Python code the response, but it does not run JavaScript or behave like a full browser.

Requests works at the HTTP response layer.

It:
- Sends an HTTP request.
- Receives a response.
- Gives access to status code, headers, and response text.
- Does not run JavaScript.
- Does not behave like a full browser.

Status code reminders:
- `200`: usually success.
- `404`: not found.
- `403`: forbidden.
- `500`: server error.


## 20) Start a Local Demo Website

To make the lesson stable, the examples use a local mini website. This avoids changes in external websites and lets us focus on HTTP requests, selectors, parsing, and browser automation.

To avoid dependency on external websites, we will serve the demo HTML files locally.

The local server lets `requests.get(...)` work with normal `http://` URLs.


In [ ]:

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass


def start_demo_server(directory: Path):
    port = find_free_port()
    handler = lambda *args, **kwargs: QuietHandler(*args, directory=str(directory), **kwargs)
    server = ThreadingHTTPServer(("127.0.0.1", port), handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server, f"http://127.0.0.1:{port}"


demo_server, demo_base_url = start_demo_server(DEMO_DIR)
static_url = f"{demo_base_url}/products_static.html"
dynamic_url = f"{demo_base_url}/products_dynamic.html"

print("Static demo:", static_url)
print("Dynamic demo:", dynamic_url)


## 21) Requests + Beautiful Soup

Requests and Beautiful Soup work well together for static pages. Requests downloads the HTML text, and Beautiful Soup turns that text into an object that can be searched with tags and selectors.

Requests fetches static HTML. Beautiful Soup parses HTML.

Best for:
- Static pages where data is already present in the HTML.
- HTML tables.
- Lists of links.
- Pages that do not require clicks, scrolling, or JavaScript-rendered data.


In [ ]:

import requests
from bs4 import BeautifulSoup

response = requests.get(static_url, timeout=10)
print("Status:", response.status_code)
print(response.text[:300])

soup = BeautifulSoup(response.text, "html.parser")
titles = [h2.get_text(strip=True) for h2 in soup.find_all("h2")]
print(titles)


## 22) Beautiful Soup: Parsing HTML

Beautiful Soup is a parser, not a browser. It is excellent for searching HTML text and extracting values, but it cannot click buttons, scroll a page, log in, or wait for JavaScript to render new content.

Beautiful Soup converts raw HTML into a searchable Python object.

It is useful for:
- Finding elements.
- Extracting text.
- Reading attributes.

It does not:
- Click.
- Scroll.
- Log in like a browser.
- Execute JavaScript.


In [ ]:

html = """
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
</div>
"""

soup = BeautifulSoup(html, "html.parser")

name = soup.select_one("h2").get_text(strip=True)
price = soup.select_one(".price").get_text(strip=True)

print(name, price)


## 23) Beautiful Soup Search Methods

Beautiful Soup gives you several ways to find elements. The `find()` methods are specific to Beautiful Soup, while `select()` and `select_one()` use CSS selectors that also transfer to Selenium.

```python
soup.find("h1")
soup.find_all("a")
soup.find("div", class_="product-card")
soup.select(".product-card")
soup.select_one(".product-card .price")
```

- `find()` and `find_all()` are Beautiful Soup-specific.
- `select()` and `select_one()` use CSS selectors.
- For beginners, CSS selectors are useful because they transfer to Selenium.


In [ ]:

soup = BeautifulSoup(response.text, "html.parser")

print("First h1:", soup.find("h1").get_text(strip=True))
print("All links:", [a.get("href") for a in soup.find_all("a")])
print("Product cards:", len(soup.select(".product-card")))
print("First price:", soup.select_one(".product-card .price").get_text(strip=True))


## 24) Extracting Repeated Items with Beautiful Soup

Repeated records should be extracted one parent container at a time. If a page has product cards, loop through the cards and extract the title, price, and link from inside each card.

Teaching point: always iterate over the repeated parent container, then extract child fields from each container.


In [ ]:

products = []

for card in soup.select(".product-card"):
    name = card.select_one("h2").get_text(strip=True)
    price = card.select_one(".price").get_text(strip=True)
    relative_url = card.select_one("a.product-link").get("href")
    product_id = card.get("data-product-id")

    products.append({
        "product_id": product_id,
        "name": name,
        "price": price,
        "url": urljoin(static_url, relative_url),
    })

products_df = pd.DataFrame(products)
products_df


## 25) Handling Missing Elements

Real websites are not always consistent. A record may be missing a price, image, rating, or link, so extraction code should check whether an element exists before trying to read from it.

Websites are inconsistent. Some records may not have all fields.

Bad version:

```python
price = card.select_one(".price").get_text(strip=True)
```

Safer version:

```python
price_element = card.select_one(".price")
price = price_element.get_text(strip=True) if price_element else None
```


In [ ]:

def get_text_or_none(element, selector):
    selected = element.select_one(selector)
    if selected:
        return selected.get_text(strip=True)
    return None


safe_products = []

for card in soup.select(".product-card"):
    safe_products.append({
        "name": get_text_or_none(card, "h2"),
        "price": get_text_or_none(card, ".price"),
        "stock": get_text_or_none(card, ".stock"),
    })

safe_products


## 26) Requests With Headers

Headers are metadata sent with an HTTP request. They can describe the client making the request, but they should not be used as a way to bypass access controls or ignore website policies.

Some websites expect browser-like headers.

```python
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
```

Headers provide metadata about the request. They should not be used to bypass access controls or policies.


In [ ]:

headers = {"User-Agent": "PythonAcademy-Lesson/1.0"}
response_with_headers = requests.get(static_url, headers=headers, timeout=10)

if response_with_headers.status_code == 200:
    print("Request successful")
else:
    print(f"Request failed: {response_with_headers.status_code}")


## 27) Selenium: Browser Automation

Selenium controls a real browser session. It is useful when you need the rendered DOM, JavaScript execution, clicks, typing, scrolling, or form interaction.

Selenium works at the browser interaction and rendered DOM layer.

It can:
- Open pages.
- Click buttons.
- Fill forms.
- Wait for content.
- Scroll.
- Extract rendered content.
- Handle dynamic JavaScript pages.


In [ ]:

# Basic Selenium example.
# If this cell fails, check that selenium is installed and Chrome is available.

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")

    driver = webdriver.Chrome(options=options)
    driver.get(static_url)

    title = driver.find_element(By.TAG_NAME, "h1").text
    print(title)

    driver.quit()
except Exception as exc:
    print("Selenium demo could not run in this environment.")
    print(type(exc).__name__, exc)


## 28) Selenium WebDriver, Driver, and WebElement

Selenium code usually revolves around a few objects: the browser controller, the elements found on the page, and locator strategies that tell Selenium how to find those elements.

Main Selenium objects:
- **WebDriver:** controls the browser.
- **driver:** the Python object connected to the browser.
- **WebElement:** one element found on the page.
- **By:** defines how to locate an element.

```python
driver = webdriver.Chrome()
driver.get("https://example.com")

element = driver.find_element(By.CSS_SELECTOR, "h1")
print(element.text)

driver.quit()
```

`driver.quit()` matters because it closes the browser session.


## 29) Finding One Element vs Many Elements

Selenium has separate methods for finding one element and many elements. This distinction matters because repeated records, such as product cards, should usually be collected as a list.

```python
driver.find_element(...)
driver.find_elements(...)
```

| Method | Behavior |
| --- | --- |
| `find_element()` | Returns one element |
| `find_elements()` | Returns a list |
| `find_element()` if nothing is found | Raises an exception |
| `find_elements()` if nothing is found | Returns an empty list |


In [ ]:

# Selenium repeated-parent pattern.

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")

    driver = webdriver.Chrome(options=options)
    driver.get(static_url)

    cards = driver.find_elements(By.CSS_SELECTOR, ".product-card")

    for card in cards:
        name = card.find_element(By.CSS_SELECTOR, "h2").text
        price = card.find_element(By.CSS_SELECTOR, ".price").text
        print(name, price)

    driver.quit()
except Exception as exc:
    print("Selenium demo could not run in this environment.")
    print(type(exc).__name__, exc)


## 30) Selenium Actions: Click, Type, Read

Browser automation becomes useful when the script can perform the same actions as a user: type into inputs, click buttons, read visible text, and read attributes such as URLs.

```python
search_box = driver.find_element(By.ID, "search")
search_box.send_keys("laptop")

button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
button.click()

title = driver.find_element(By.TAG_NAME, "h1").text
```

- `send_keys()` types into inputs.
- `click()` clicks buttons or links.
- `.text` reads visible text.
- `.get_attribute("href")` reads attributes.


## 31) Waiting for Elements

Dynamic pages often need time before the target element exists or becomes clickable. Explicit waits make Selenium scripts more reliable because they wait for a specific condition instead of pausing blindly.

Dynamic pages need time to load. Avoid relying on `time.sleep()` as the main strategy.

Better pattern:

```python
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)

price = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".price"))
)
```

Explicit waits wait until a condition is true, making scripts more reliable.


In [ ]:

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    driver.get(dynamic_url)

    cards = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".product-card"))
    )

    print("Cards loaded:", len(cards))
    print([card.find_element(By.CSS_SELECTOR, "h2").text for card in cards])

    driver.quit()
except Exception as exc:
    print("Selenium wait demo could not run in this environment.")
    print(type(exc).__name__, exc)


## 32) Common Selenium Wait Conditions

Not all waiting conditions mean the same thing. An element can exist in the DOM but still be hidden or not ready to click, so the wait condition should match what the script needs to do next.

```python
EC.presence_of_element_located((By.CSS_SELECTOR, ".price"))
EC.visibility_of_element_located((By.CSS_SELECTOR, ".price"))
EC.element_to_be_clickable((By.CSS_SELECTOR, "button"))
```

- **Presence:** exists in the DOM.
- **Visibility:** exists and is visible.
- **Clickable:** visible and enabled for clicking.


## 33) Static vs Dynamic Pages

A static page already contains the target data in the HTML response. A dynamic page depends on JavaScript or later network calls. Diagnosing this difference is one of the most important scraping skills.

Static page:
- Data appears in `requests.get(url).text`.
- Beautiful Soup can parse it.

Dynamic page:
- Data does not appear in raw HTML.
- Data appears only after JavaScript runs.
- Selenium may be needed.
- Sometimes the better option is to inspect Network tab and find the background API.

Suggested workflow:
1. Try API first.
2. Try Requests + Beautiful Soup.
3. Use Selenium only when browser interaction or JavaScript rendering is required.


In [ ]:

static_response = requests.get(static_url, timeout=10)
dynamic_response = requests.get(dynamic_url, timeout=10)

print("Static raw HTML contains product-card:", "product-card" in static_response.text)
print("Dynamic raw HTML contains product-card:", "product-card" in dynamic_response.text)
print("Dynamic raw HTML contains Wireless Mouse:", "Wireless Mouse" in dynamic_response.text)

# The dynamic page includes product data inside JavaScript in this local demo,
# but not as rendered HTML product cards. Real sites vary.


## 34) DevTools Network Tab

The Network tab can reveal where the browser gets its data. Sometimes the page looks complicated, but the data comes from a clean JSON endpoint that is easier to request directly.

The Network tab shows requests made by the browser.

Students can use it to detect:
- API endpoints.
- JSON responses.
- Pagination requests.
- Search/filter requests.
- Authentication-related calls.

Teaching point: sometimes the page looks complicated, but the data comes from a simple JSON endpoint.


## 35) Decision Tree: Which Tool Should I Use?

Tool choice should follow diagnosis. Start by checking for an API, then check whether the data is in the initial HTML, then inspect background requests, and use Selenium only when browser behavior is actually needed.

```text
Is there an official API?
  yes -> Use API with Requests
  no  -> Continue

Is the data present in the initial HTML?
  yes -> Use Requests + Beautiful Soup
  no  -> Continue

Is the data loaded by background JSON requests?
  yes -> Use Requests against the JSON endpoint, if allowed
  no  -> Continue

Do you need clicks, login, scrolling, or JavaScript rendering?
  yes -> Use Selenium
  no  -> Reinspect the page

Is scraping allowed and ethical?
  yes -> Build carefully with rate limits and logging
  no  -> Do not scrape
```


## 36) Selenium vs Beautiful Soup

Requests, Beautiful Soup, and Selenium operate at different layers. Requests gets responses, Beautiful Soup parses HTML, and Selenium interacts with the browser-rendered DOM.

| Tool or pattern | Strength | Tradeoff | Use when |
| --- | --- | --- | --- |
| Requests | Fast HTTP access | No JavaScript or clicks | API, static HTML, files |
| Beautiful Soup | Simple HTML parsing | Needs HTML text first | Data is already in HTML |
| Selenium | Real browser behavior | Slower and heavier | Dynamic pages, clicks, forms, login |
| Selenium + Beautiful Soup | Render then parse | More moving parts | Need browser-rendered HTML plus parser convenience |


## 37) Combined Selenium + Beautiful Soup Pattern

Sometimes the best pattern is to use Selenium only for rendering and waiting, then pass the final HTML to Beautiful Soup for parsing. This keeps interaction and extraction responsibilities separate.

Pattern:
1. Open page with Selenium.
2. Wait for content to load.
3. Get `driver.page_source`.
4. Parse with Beautiful Soup.


In [ ]:

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from bs4 import BeautifulSoup

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")

    driver = webdriver.Chrome(options=options)
    driver.get(dynamic_url)

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".product-card"))
    )

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    data = [item.get_text(strip=True) for item in soup.select(".product-card h2")]

    driver.quit()
    print(data)
except Exception as exc:
    print("Combined Selenium + Beautiful Soup demo could not run in this environment.")
    print(type(exc).__name__, exc)


## 38) Designing a Web Ingestion Workflow

A web ingestion workflow should be designed before it is coded. The extractor is only one piece; the workflow also needs validation, raw data storage, transformation, output storage, logging, and error handling.

```text
1. Define target source
2. Check terms of use and robots.txt
3. Inspect page/API
4. Extract data
5. Validate data
6. Save raw data
7. Transform data
8. Load into CSV, database, or data lake
9. Log results
10. Notify if errors occur
```

```text
Website -> Selenium/Requests -> Parser -> Validation -> CSV/Database -> Report/Notification
```


## 39) Data Ingestion Best Practices

Good ingestion workflows are built to be rerun and debugged. Saving raw inputs, separating steps, validating outputs, and logging results make the difference between a demo and something maintainable.

- Save raw HTML or raw JSON when possible.
- Separate extraction, transformation, and loading logic.
- Use a clear folder structure.
- Add logging.
- Add retries.
- Handle missing values.
- Avoid hardcoding fragile selectors.
- Make the workflow restartable.


In [ ]:

raw_html_path = RAW_DIR / "products_static.html"
processed_csv_path = PROCESSED_DIR / "products.csv"

raw_html_path.write_text(response.text, encoding="utf-8")
products_df.to_csv(processed_csv_path, index=False)

logger.info("Saved raw HTML to %s", raw_html_path)
logger.info("Saved processed CSV to %s", processed_csv_path)

pd.read_csv(processed_csv_path)


## 40) Error Handling and Retries

Failures are normal in web automation. Servers can time out, selectors can change, pages can return empty responses, and authentication can expire. The workflow should fail clearly or retry only when retrying is safe.

Common failures:
- Website unavailable.
- Selector changed.
- Timeout.
- Empty response.
- Authentication expired.
- Duplicate records.

A simple retry strategy:
- Retry temporary failures a limited number of times.
- Log every attempt.
- Stop and raise a clear error when the workflow cannot continue safely.


In [ ]:

import time

def fetch_with_retries(url, attempts=3, delay_seconds=1):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            logger.info("Fetching %s (attempt %s/%s)", url, attempt, attempts)
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            logger.warning("Fetch failed: %s", exc)
            if attempt < attempts:
                time.sleep(delay_seconds)

    raise RuntimeError(f"Could not fetch {url}") from last_error


retry_response = fetch_with_retries(static_url)
print(retry_response.status_code, len(retry_response.text))


## 41) Scheduling and Orchestration

A script becomes more useful when it can run at the right time without manual effort. Scheduling is enough for simple jobs; orchestration becomes useful when tasks have dependencies, retries, monitoring, and backfills.

Simple options:
- Run manually.
- cron.
- Windows Task Scheduler.

Advanced options:
- Airflow.
- Prefect.
- Dagster.
- Cloud workflow tools.

Orchestration becomes necessary when you need scheduled execution, task dependencies, retries, logs, alerting, backfills, and visibility for other people.


## 42) Ethical and Legal Considerations

Scraping is both a technical and policy decision. Publicly visible data is not automatically free to scrape in any way, so responsible automation starts with permissions, rate limits, and respect for access controls.

- Respect website terms of service.
- Prefer APIs when available.
- Avoid aggressive scraping.
- Use rate limits.
- Do not scrape private or protected data without authorization.
- Avoid bypassing access controls.
- Identify yourself when appropriate.

Scraping public pages is not automatically safe or allowed. Treat scraping as an engineering and governance decision.


## 43) Common Beginner Mistakes

Most beginner scraping bugs come from choosing the tool too early or trusting selectors too quickly. Inspect first, choose the simplest layer, handle missing data, and make the script observable.

- Scraping what is visible instead of inspecting the DOM.
- Using Selenium for everything.
- Using Beautiful Soup on JavaScript-rendered pages and getting empty results.
- Selecting elements with fragile CSS paths.
- Ignoring status codes.
- Not handling missing values.
- Not using waits in Selenium.
- Not closing the browser.
- Not respecting rate limits or terms of service.
- Mixing data from different cards because parent containers were not used.


## 44) Practical Checklist Before Scraping

Before writing code, walk through the checklist below. It helps convert “I can see the data” into a safer question: “Can I collect this data reliably and responsibly?”

1. Is there an official API?
2. Is scraping allowed?
3. Is the data public?
4. Is the page static or dynamic?
5. Is the target data in HTML, attributes, or JSON?
6. What is the repeated parent container?
7. What selectors are stable?
8. What happens if a field is missing?
9. Do we need pagination, scrolling, or login?
10. How will we validate the extracted data?
11. How will we store raw and processed data?
12. How will we log errors?
13. How will we avoid overloading the website?


In [ ]:

# Exercise 1 TODO

# response = requests.get(___, timeout=10)
# response.raise_for_status()
# soup = BeautifulSoup(___, "html.parser")

exercise_1_products = []

# for card in soup.select("___"):
#     name = card.select_one("___").get_text(strip=True)
#     price = card.select_one("___").get_text(strip=True)
#     url = card.select_one("___").get("href")
#
#     exercise_1_products.append({
#         "name": name,
#         "price": price,
#         "url": urljoin(static_url, url),
#     })

exercise_1_df = pd.DataFrame(exercise_1_products)
exercise_1_df


In [ ]:

# Run after completing Exercise 1.

assert len(exercise_1_df) == 3
assert {"name", "price", "url"}.issubset(exercise_1_df.columns)
assert "Wireless Mouse" in set(exercise_1_df["name"])
assert exercise_1_df["url"].str.startswith("http://127.0.0.1").all()


In [ ]:

html = """
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a class="product-link" href="/products/mouse">View details</a>
</div>

<div class="product-card">
  <h2>Keyboard</h2>
  <p class="price">$45.00</p>
  <a class="product-link" href="/products/keyboard">View details</a>
</div>
"""

soup = BeautifulSoup(html, "html.parser")

products = []

for card in soup.select("___"):
    name = card.select_one("___").get_text(strip=True)
    price = card.select_one("___").get_text(strip=True)
    url = card.select_one("___").get("href")

    products.append({
        "name": name,
        "price": price,
        "url": url,
    })

print(products)


In [ ]:

# Exercise 2 TODO

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC

selenium_results = []

# options = Options()
# options.add_argument("--headless=new")
# options.add_argument("--window-size=1200,900")
#
# driver = webdriver.Chrome(options=options)
# wait = WebDriverWait(driver, 10)
#
# try:
#     driver.get(dynamic_url)
#
#     search_box = wait.until(
#         EC.visibility_of_element_located((By.CSS_SELECTOR, "___"))
#     )
#     search_box.send_keys("keyboard")
#
#     button = wait.until(
#         EC.element_to_be_clickable((By.CSS_SELECTOR, "___"))
#     )
#     button.click()
#
#     cards = wait.until(
#         EC.presence_of_all_elements_located((By.CSS_SELECTOR, "___"))
#     )
#
#     for card in cards:
#         selenium_results.append({
#             "name": card.find_element(By.CSS_SELECTOR, "___").text,
#             "price": card.find_element(By.CSS_SELECTOR, "___").text,
#             "url": card.find_element(By.CSS_SELECTOR, "___").get_attribute("href"),
#         })
# finally:
#     driver.quit()

selenium_results_df = pd.DataFrame(selenium_results)
selenium_results_df


In [ ]:

project_root = BASE_DIR / "web_ingestion_project"
for path in [
    project_root / "data" / "raw",
    project_root / "data" / "processed",
    project_root / "logs",
    project_root / "src",
]:
    path.mkdir(parents=True, exist_ok=True)

for filename in ["extract.py", "transform.py", "load.py", "main.py"]:
    target = project_root / "src" / filename
    if not target.exists():
        target.write_text(f"# {filename}\n", encoding="utf-8")

(project_root / "requirements.txt").write_text("requests\nbeautifulsoup4\npandas\n", encoding="utf-8")
(project_root / "README.md").write_text("# Web Ingestion Project\n", encoding="utf-8")

print(project_root)


## 45) Key Vocabulary

These terms appear throughout web automation work. Treat them as shared vocabulary for discussing what the browser sees, what Python receives, and how code locates data.

| Term | Meaning |
| --- | --- |
| HTML | Language used to structure web pages |
| Element | One HTML object, such as a heading, paragraph, link, or button |
| Tag | Element name, such as `div`, `a`, or `h1` |
| Attribute | Extra information inside a tag, such as `href`, `src`, `class`, or `id` |
| Text content | Visible text inside an element |
| DOM | Browser's live representation of the web page |
| Selector | Pattern used to locate elements |
| CSS selector | Selector syntax commonly used in scraping and automation |
| XPath | Path-like selector syntax for locating elements |
| Static page | Page where data is present in the initial HTML |
| Dynamic page | Page where JavaScript loads or changes content |
| WebDriver | Selenium object that controls the browser |
| WebElement | Selenium object representing an element on the page |
| Explicit wait | Selenium strategy for waiting until a condition is true |
| Parser | Tool that reads structured text like HTML |
| Scraping | Extracting information from web pages |
| Ingestion | Collecting data and bringing it into a system for storage or processing |


## 46) Summary

The main habit is to diagnose first and automate second. Once you know where the data lives, choose the simplest appropriate tool and build the workflow with validation, logging, and clear outputs.

- Automation is about repeatable, reliable execution.
- Web scraping is one ingestion method, not always the best one.
- Use APIs first when possible.
- Use Requests at the HTTP response layer.
- Use Beautiful Soup at the HTML parsing layer.
- Use Selenium at the browser interaction and rendered DOM layer.
- Diagnose the page first, then choose the simplest appropriate tool.
- Build workflows with logging, retries, validation, and clear outputs.


## 47) Discussion Questions

Use these questions to connect the notebook examples to production decisions, team practices, and responsible automation.

- When should we avoid scraping?
- When is Selenium overkill?
- What makes an automation production-ready?
- How do we detect if a website changed?
- How would we monitor this workflow?


## 48) Optional Advanced Topics

After the basics are comfortable, these topics extend the same mental model into deployment, monitoring, and production data engineering.

- Headless browsers.
- Dockerizing scraping jobs.
- Proxy and rate-limit considerations.
- Airflow DAGs.
- Data quality checks.
- Incremental ingestion.
- Change data capture from websites.
- Cloud deployment.


> Content created by [**Carlos Cruz-Maldonado**](https://www.linkedin.com/in/carloscruzmaldonado/).
> I am available to answer any questions or provide further assistance.
> Feel free to reach out to me at any time.


In [ ]:

# Exercise A TODO

card_selector = ""
name_selector = ""
price_selector = ""
link_selector = ""


In [ ]:

# Exercise A Checks

assert card_selector == ".product-card", "Use the repeated parent container selector."
assert name_selector == "h2", "The product name is inside the h2 within each card."
assert price_selector == ".price", "The price element has class='price'."
assert link_selector == "a.product-link", "The link is an anchor with class='product-link'."

print("Exercise A passed")


In [ ]:

# Exercise B TODO

from bs4 import BeautifulSoup

exercise_b_html = """
<div class="product-card">
  <h2>Wireless Mouse</h2>
  <p class="price">$25.99</p>
  <a class="product-link" href="/products/mouse">View details</a>
</div>

<div class="product-card">
  <h2>Keyboard</h2>
  <p class="price">$45.00</p>
  <a class="product-link" href="/products/keyboard">View details</a>
</div>
"""

exercise_b_soup = BeautifulSoup(exercise_b_html, "html.parser")
exercise_b_products = []

# TODO: loop over product cards and append dictionaries to exercise_b_products


In [ ]:

# Exercise B Checks

assert isinstance(exercise_b_products, list), "exercise_b_products must be a list."
assert len(exercise_b_products) == 2, "You should extract exactly two products."
assert exercise_b_products[0]["name"] == "Wireless Mouse"
assert exercise_b_products[0]["price"] == "$25.99"
assert exercise_b_products[0]["url"] == "/products/mouse"
assert exercise_b_products[1]["name"] == "Keyboard"
assert set(exercise_b_products[0]) == {"name", "price", "url"}

print("Exercise B passed")


In [ ]:

# Exercise C TODO

import pandas as pd
from bs4 import BeautifulSoup


def parse_products_html(html_text):
    # TODO: parse html_text and return a DataFrame
    pass


In [ ]:

# Exercise C Checks

sample_html = (DEMO_DIR / "products_static.html").read_text(encoding="utf-8")
exercise_c_df = parse_products_html(sample_html)

assert isinstance(exercise_c_df, pd.DataFrame), "Return a pandas DataFrame."
assert list(exercise_c_df.columns) == ["product_id", "name", "price", "url"]
assert len(exercise_c_df) == 3
assert exercise_c_df.loc[0, "product_id"] == "001"
assert exercise_c_df.loc[0, "name"] == "Wireless Mouse"
assert exercise_c_df["price"].str.startswith("$").all()
assert exercise_c_df["url"].str.startswith("/products/").all()

print("Exercise C passed")
exercise_c_df


In [ ]:

# Exercise D TODO

student_products_path = None
student_products_df = None

# TODO: parse local HTML, save CSV, assign student_products_path and student_products_df


In [ ]:

# Exercise D Checks

assert student_products_path is not None, "Assign the CSV path to student_products_path."
assert Path(student_products_path).exists(), "The CSV file should exist on disk."
assert isinstance(student_products_df, pd.DataFrame), "student_products_df must be a DataFrame."
assert len(student_products_df) == 3
assert {"product_id", "name", "price", "url"}.issubset(student_products_df.columns)

loaded_student_products = pd.read_csv(student_products_path)
assert len(loaded_student_products) == 3
assert loaded_student_products["name"].tolist() == student_products_df["name"].tolist()

print("Exercise D passed")
loaded_student_products
